# fastkernels Colab kernel test

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suryavanshi/fastkernels/blob/main/notebooks/colab_kernel_test.ipynb)

Use this notebook on a Colab GPU runtime to install `fastkernels`, run the unit tests, and compare the Triton MoE building blocks against the PyTorch reference path.

Before running the cells, choose `Runtime > Change runtime type > GPU`.

## Clone and install

By default this clones `https://github.com/suryavanshi/fastkernels.git`. To test a fork or branch, set `FASTKERNELS_REPO` or `FASTKERNELS_BRANCH` before running the setup cell.

In [ ]:
import os
import pathlib
import subprocess

REPO_URL = os.environ.get("FASTKERNELS_REPO", "https://github.com/suryavanshi/fastkernels.git")
BRANCH = os.environ.get("FASTKERNELS_BRANCH", "")
REPO_DIR = pathlib.Path("/content/fastkernels")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

if BRANCH:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)

os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"Using repository at {REPO_DIR}")
print(f"Commit: {commit}")

In [ ]:
%pip install -q "torch==2.7.1" "triton==3.3.1" numpy
%pip install -q -e ".[triton,dev]"

## Runtime checks

In [ ]:
import fastkernels
import torch
import triton

print(f"fastkernels: {fastkernels.__version__}")
print(f"torch: {torch.__version__}")
print(f"triton: {triton.__version__}")
print(f"cuda: {torch.version.cuda}")

assert torch.cuda.is_available(), "Select a GPU runtime before running the kernel tests."
print(f"gpu: {torch.cuda.get_device_name(0)}")

In [ ]:
!pytest -q

## Qwen3.5 MoE shape report

In [ ]:
!python bench/qwen35_profile.py --tokens 1 16 128

## Direct Triton kernel correctness checks

In [ ]:
import torch

from fastkernels.kernels.triton import triton_expert_histogram, triton_fused_swiglu
from fastkernels.reference import reference_expert_histogram, reference_fused_swiglu
from fastkernels.testing import default_tolerance

torch.manual_seed(0)
dtype = torch.bfloat16
device = "cuda"

gate_up = torch.randn(2048, 1024, device=device, dtype=dtype)
swiglu_ref = reference_fused_swiglu(gate_up)
swiglu_out = triton_fused_swiglu(gate_up)
atol, rtol = default_tolerance(dtype)
torch.testing.assert_close(swiglu_out, swiglu_ref, atol=atol, rtol=rtol)
print("triton_fused_swiglu: correctness ok")

topk_ids = torch.randint(0, 256, (512, 8), device=device, dtype=torch.int32)
hist_ref = reference_expert_histogram(topk_ids, 256)
hist_out = triton_expert_histogram(topk_ids, 256)
torch.testing.assert_close(hist_out.cpu(), hist_ref.cpu(), atol=0, rtol=0)
print("triton_expert_histogram: correctness ok")

## Microbenchmarks

In [ ]:
!python bench/moe_microbench.py --device cuda --dtype bfloat16 --rows 2048 --tokens 4 --warmup 5 --iters 25 --skip-routed-moe

## Tiny end-to-end routed MoE reference check

This keeps the tensor sizes small enough for a quick Colab smoke test while still exercising the reference routed MoE path.

In [ ]:
!python bench/moe_microbench.py --device cuda --dtype bfloat16 --rows 512 --tokens 2 --hidden 256 --intermediate 128 --experts 16 --top-k 4 --warmup 3 --iters 10